# 📗 어댑터 관리와 DoRA

교안 01에서 에폭마다 저장한 LoRA 어댑터 4개로 어댑터 관리, DoRA, 병합을 배웁니다.

<img src="images/10_lesson02_goals.png" width="1050" alt="교안 02에서 배울 네 가지: 1 에폭별 어댑터 고르기, 2 지난 시간 풀 파인튜닝과 비교, 3 DoRA와 비교하기, 4 쓸 어댑터 정하고 합치기.">

**학습 목표**

1. **어댑터 얹고 고르기:** `PeftModel.from_pretrained`, `load_adapter`로 베이스 모델 하나에 어댑터 4개를 얹고, `set_adapter`로 바꿔 켜며 채점해 LoRA Tuned를 고릅니다.
2. **풀 파인튜닝과 비교:** 'LoRA는 덜 배우고 덜 잊는다'는 논문 결과가 이 데이터에서도 같은지 같은 지표로 봅니다.
3. **DoRA(Weight-Decomposed Low-Rank Adaptation, 가중치 분해 저랭크 적응):** 가중치를 크기와 방향으로 나눠 학습하는 방법을 LoRA Tuned와 같은 지표, 학습 파라미터 수, 학습 시간으로 비교합니다.
4. **결정과 병합:** 결정 테이블(결정, 근거, 다시 검토할 조건)로 쓸 어댑터를 정하고, `merge_and_unload`로 어댑터를 베이스 가중치에 합쳐 답과 저장 크기를 확인합니다.

## 1. 실습 준비

지난 시간과 교안 01에서 쓴 준비 코드(경로, 모델, 판사, 평가 데이터, 평가 함수, MLflow)를 한 셀에 모았습니다. **교안 01을 끝까지 실행한 뒤** 이 셀을 실행합니다.

- **API 키:** 판사는 `.env`의 `TYPESAFE_API_KEY`가 필요합니다.

In [ ]:
# ---- 라이브러리와 경로 ----
import json
import platform
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

material_dir = Path(".")  # data가 있는 단원 폴더. 정답 노트북에서는 ".."
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(exist_ok=True)

# 계산 장치: RunPod처럼 GPU(cuda)가 있으면 GPU, 없으면 CPU에서 계산합니다
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("계산 장치:", device, "|", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("계산 장치:", device)

# 테이블의 긴 답변이 잘리지 않게 칸의 글자 수 제한을 풉니다
pd.set_option("display.max_colwidth", None)

# 그래프 글꼴: 영문, 숫자, 기호는 DejaVu Sans, 한글은 운영체제의 한글 글꼴이 맡습니다
from matplotlib import font_manager

if platform.system() == "Windows":
    korean_font = "Malgun Gothic"
elif platform.system() == "Darwin":
    korean_font = "AppleGothic"
else:
    korean_font = "NanumGothic"
    # 리눅스(RunPod): 글꼴을 설치하기 전에 만든 matplotlib 글꼴 목록에는 나눔고딕이 없으므로, 설치된 파일을 직접 등록합니다
    for font_path in font_manager.findSystemFonts():
        if "NanumGothic" in font_path:
            font_manager.fontManager.addfont(font_path)
plt.rcParams["font.family"] = ["DejaVu Sans", korean_font]
plt.rcParams["axes.unicode_minus"] = False  # 음수 부호가 네모로 깨지지 않게 합니다

# ---- 학습 전 모델 ----
MODEL_NAME = "Qwen/Qwen3.5-0.8B"

# 토크나이저와 학습 전 모델(Base)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32).to(device)

# ---- 답 생성과 데이터 변환 ----
# 지난 시간 데이터의 시스템 메시지와 같은 문장
SYSTEM_PROMPT = "너는 노무 상담 챗봇이다. 노동법 조문에 근거해 존댓말로 간결히 답한다."

# 대화 끝 토큰 <|im_end|>의 id. 생성이 이 토큰에서 멈추도록 eos_token_id로 넘깁니다
END_OF_TURN_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")


def generate_replies(model, questions, system_prompt, max_new_tokens=256):
    """질문을 32개씩 묶어 생성하고, 질문 순서대로 답 목록을 돌려줍니다. system_prompt=None 이면 시스템 메시지 없이 묻습니다."""
    # 여러 질문을 묶으면 짧은 질문을 패딩으로 채워 길이를 맞춥니다. 답은 오른쪽 끝에 이어 쓰므로 패딩은 왼쪽에 둡니다
    tokenizer.padding_side = "left"
    answers = []
    for start in range(0, len(questions), 32):
        # generate_reply 와 같은 chat template으로 질문을 감쌉니다
        prompts = []
        for question in questions[start:start + 32]:
            messages = []
            if system_prompt is not None:
                messages.append({"role": "system", "content": system_prompt})
            messages.append({"role": "user", "content": question})
            prompts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False))
        inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)

        # generate_reply 와 같은 그리디 생성
        with torch.no_grad():
            output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                        eos_token_id=END_OF_TURN_ID, pad_token_id=tokenizer.pad_token_id)

        # 입력 길이 뒤의 새 토큰만 글자로 바꿉니다(먼저 끝난 답 뒤의 패딩은 skip_special_tokens 로 빠집니다)
        for new_ids in output_ids[:, inputs["input_ids"].shape[1]:]:
            answers.append(tokenizer.decode(new_ids, skip_special_tokens=True))

    tokenizer.padding_side = "right"  # 학습기가 쓰는 오른쪽 패딩으로 되돌립니다
    return answers


def load_jsonl(path):
    """JSONL 파일을 한 줄에 딕셔너리 하나씩 읽어 리스트로 돌려줍니다(지난 시간과 같은 함수)."""
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


def to_prompt_completion(row):
    """messages를 앞부분(system이 있으면 system, user)과 배울 답(마지막 assistant)으로 나눕니다."""
    prompt = row["messages"][:-1]      # 마지막 답 앞의 메시지(system, user)
    completion = row["messages"][-1:]  # 마지막 assistant 메시지 하나
    return {"prompt": prompt, "completion": completion}

# ---- 판사 ----
import os

from dotenv import load_dotenv

# 단원 폴더의 .env를 먼저 읽고, 없는 키만 상위 실습자료 폴더의 .env에서 채웁니다
load_dotenv(material_dir / ".env")
load_dotenv(material_dir.resolve().parent / ".env")

os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "1"  # 사용 기록을 외부로 보내지 않음(deepeval을 불러오기 전에 설정)
from deepeval import evaluate
from deepeval.evaluate import AsyncConfig, DisplayConfig
from deepeval.metrics import JevEval
from deepeval.metrics.jev_eval import Noul
from deepeval.test_case import LLMTestCase, SingleTurnParams

# 판정 결과가 모델 버전에 따라 달라지지 않게 Jev 버전을 고정합니다
jev_model_name = os.getenv("TYPESAFE_MODEL", "jev-1.13.0")
print("Jev 모델:", jev_model_name)
print("Jev 키 읽음:", os.getenv("TYPESAFE_API_KEY") is not None)

# 두 판사가 함께 쓰는 판정 문장. Jev가 문장마다 참일 확률을 돌려줍니다
BASE_STATEMENTS = [
    Noul("actual_output은 input의 질문에 직접 답한다."),
    Noul("actual_output의 사실(숫자, 기간, 조건, 이름)과 결론이 expected_output과 일치한다. 표현이 달라도 같은 정보를 전하면 일치로 본다."),
    Noul("actual_output은 input의 질문과 관계없는 지어낸 내용을 덧붙이지 않는다."),
    Noul("actual_output은 같은 문장이나 구절을 되풀이하지 않는다."),
]

# 일반 질문과 학습 데이터 표본용: 질문, 모델의 답, 정답 답변을 봅니다
general_metric = JevEval(
    name="정확성",
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.EXPECTED_OUTPUT],  # 판정 문장에서 가리키는 필드
    questions=BASE_STATEMENTS,
    system_one_model=jev_model_name,
    strict_mode=True,  # 모든 문장의 참일 확률이 0.5 이상이어야 통과
    async_mode=True,  # Jev 호출을 비동기로 보냄(DeepEval 기본값)
)

# 노무, 전월세 질문용: 근거 조문(context)도 보고, 정답 답변에 없는 말이 조문과 어긋나지 않는지 한 문장 더 묻습니다
law_metric = JevEval(
    name="정확성(근거 조문)",
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.EXPECTED_OUTPUT, SingleTurnParams.CONTEXT],
    questions=BASE_STATEMENTS + [
        Noul("actual_output이 expected_output에 없는 내용을 말할 때, 그 내용은 context(근거 조문)와 모순되지 않는다."),
    ],
    system_one_model=jev_model_name,
    strict_mode=True,
    async_mode=True,
)


def judge_answers(rows, answers):
    """판사가 답 목록을 동시에 채점해, 문항 순서대로 점수, 통과, 이유 열을 가진 테이블을 돌려줍니다."""
    # rows 는 모두 같은 종류입니다. 근거 조문(law_text)이 있으면 근거 조문 기준으로 채점합니다
    metric = general_metric
    if "law_text" in rows[0]:
        metric = law_metric

    # 문항마다 테스트 케이스 하나
    test_cases = []
    for index, row in enumerate(rows):
        context = None
        if "law_text" in row:
            context = [row["law_text"]]
        test_cases.append(LLMTestCase(
            name=str(index),                          # 결과를 문항 순서로 되돌리는 번호(동시 채점은 끝나는 순서가 섞입니다)
            input=row["question"],                    # 질문
            actual_output=answers[index],             # 채점할 모델의 답
            expected_output=row["reference_answer"],  # 정답 답변
            context=context,                          # 근거 조문 원문(없으면 None)
        ))

    # 판사를 동시에 부릅니다. 동시에 보낼 문항 수 상한은 64이고, 반 전체가 같은 키를 쓰다 429(요청 한도 초과)가 나면 줄입니다
    result = evaluate(test_cases=test_cases, metrics=[metric], async_config=AsyncConfig(max_concurrent=64),
                      display_config=DisplayConfig(print_results=False, show_indicator=False))  # 긴 기본 출력을 끕니다

    # 결과를 번호(name)로 찾아 문항 순서대로 놓습니다. 점수는 모든 문장이 참이면 1, 아니면 0이고, 이유에는 문장마다 참일 확률이 나옵니다
    judged = {}
    for test_result in result.test_results:
        judged[test_result.name] = test_result.metrics_data[0]  # 지표를 하나만 넘겼습니다
    judged_rows = []
    for index in range(len(rows)):
        metric_data = judged[str(index)]
        judged_rows.append({"점수": metric_data.score, "통과": metric_data.success, "이유": metric_data.reason})
    return pd.DataFrame(judged_rows)

# ---- 평가 데이터 ----
# 평가 파일: 노무 평가셋(재현, 적용), 범위 밖 질문, 일반 질문, 일반 문단
eval_rows = load_jsonl(data_dir / "본실습" / "labor_eval.jsonl")
out_of_scope_rows = load_jsonl(data_dir / "본실습" / "out_of_scope_eval.jsonl")
general_rows = load_jsonl(data_dir / "본실습" / "general_questions.jsonl")
general_texts = json.loads((data_dir / "본실습" / "general_texts.json").read_text(encoding="utf-8"))

# 학습 데이터 표본: 거절 답을 뺀 학습 행에서 같은 간격으로 고른 40행(지난 시간과 같은 행)
train_rows = load_jsonl(data_dir / "본실습" / "labor_train.jsonl")
answer_rows = []
for row in train_rows:
    if "답할 수 없" not in row["answer"]:
        answer_rows.append(row)
sample_gap = len(answer_rows) // 40
train_sample_rows = []
for row in answer_rows[::sample_gap][:40]:
    # 학습한 답(answer)을 정답 답변으로 씁니다
    train_sample_rows.append({"question": row["question"], "reference_answer": row["answer"]})

# ---- 평가 함수 ----
def answer_and_judge(model, rows, system_prompt):
    """rows의 질문에 답을 생성하고 채점해, 질문마다 한 행씩 모은 테이블을 돌려줍니다."""
    questions = []
    for row in rows:
        questions.append(row["question"])
    answers = generate_replies(model, questions, system_prompt)

    # 거절은 거절 키워드가 답에 있는지로 셉니다(거절률과 과잉 거절률에 씁니다)
    result_rows = []
    for row, answer in zip(rows, answers):
        result_rows.append({"질문": row["question"], "종류": row.get("question_type", ""), "답변": answer,
                            "거절": "답할 수 없" in answer})
    result_table = pd.DataFrame(result_rows)

    # 정답 답변이 있는 행만 판사가 채점합니다(범위 밖 질문은 정답 답변이 없습니다)
    if "reference_answer" in rows[0]:
        result_table = pd.concat([result_table, judge_answers(rows, answers)], axis=1)
    return result_table

import math


def perplexity(model, texts):
    """문단 목록 전체의 perplexity: 예측한 토큰 전체의 평균 손실에 exp를 합니다."""
    total_loss = 0.0
    total_tokens = 0
    for text in texts:
        input_ids = tokenizer(text, return_tensors="pt").input_ids.to(model.device)
        with torch.no_grad():
            loss = model(input_ids, labels=input_ids).loss.item()  # 입력을 정답으로 주면 다음 토큰 예측 손실의 평균을 돌려줍니다

        # 첫 토큰은 예측하지 않으므로 예측한 토큰 수는 길이 - 1입니다. 평균 * 개수 = 이 문단의 손실 합
        n_predicted = input_ids.shape[1] - 1
        total_loss = total_loss + loss * n_predicted
        total_tokens = total_tokens + n_predicted
    return math.exp(total_loss / total_tokens)


# 지표 이름(MLflow 에 기록할 이름)과 테이블에 보일 이름. 학습 데이터 정답률을 맨 앞에 둡니다
METRIC_NAMES = {"train_acc": "학습 데이터 정답률", "domain_recall_acc": "도메인 정답률(재현)", "domain_apply_acc": "도메인 정답률(적용)",
                "over_refusal_rate": "과잉 거절률", "refusal_rate": "거절률", "general_acc": "일반 질문 정답률", "general_ppl": "일반 perplexity"}


def evaluate_model(model):
    """모델 하나를 앞의 평가 네 가지와 perplexity 로 평가해 사전으로 돌려줍니다."""
    return {
        "train": answer_and_judge(model, train_sample_rows, SYSTEM_PROMPT),
        "domain": answer_and_judge(model, eval_rows, SYSTEM_PROMPT),
        "refusal": answer_and_judge(model, out_of_scope_rows, SYSTEM_PROMPT),
        "general": answer_and_judge(model, general_rows, None),
        "ppl": perplexity(model, general_texts),
    }


def table_metrics(tables):
    """evaluate_model 이 돌려준 사전으로 지표 일곱 개를 계산합니다. 정답률, 거절률은 True 를 1로 본 평균입니다."""
    domain_table = tables["domain"]
    return {
        "train_acc": tables["train"]["통과"].mean(),
        "domain_recall_acc": domain_table[domain_table["종류"] == "재현"]["통과"].mean(),
        "domain_apply_acc": domain_table[domain_table["종류"] == "적용"]["통과"].mean(),
        "over_refusal_rate": domain_table["거절"].mean(),
        "refusal_rate": tables["refusal"]["거절"].mean(),
        "general_acc": tables["general"]["통과"].mean(),
        "general_ppl": tables["ppl"],
    }

# ---- MLflow ----
import mlflow

# 실험 기록은 output/mlflow.db 한 파일에 남깁니다(별도 서버 없이 sqlite)
mlflow.set_tracking_uri(f"sqlite:///{output_dir / 'mlflow.db'}")
mlflow.set_experiment("labor-lora")
print("MLflow 기록 위치:", mlflow.get_tracking_uri())

# ---- 준비 확인 ----
print("Base:", MODEL_NAME, "| 노무 평가셋:", len(eval_rows), "| 범위 밖:", len(out_of_scope_rows), "| 일반 질문:", len(general_rows),
      "| 일반 문단:", len(general_texts), "| 학습 데이터 표본:", len(train_sample_rows))

## 2. 에폭별 어댑터 얹기와 고르기

### 2-1. Base 지표: 지난 시간 값

Base는 지난 시간과 같은 모델, 같은 질문, 같은 그리디 생성이라 답이 같습니다. 그래서 다시 채점하지 않고 지난 시간 지표를 `Base(지난 시간)` 행으로 씁니다.

- **값:** 아래 사전은 지난 시간 지표 테이블(`summary_table`) `Base` 행의 강사 실행 값입니다. 자기 값이 있으면 바꿔 넣습니다.

In [ ]:
# 지난 시간 지표 테이블(summary_table)의 Base 행입니다(강사 실행 값, 자기 값이 있으면 바꿔 넣습니다)
# 키는 METRIC_NAMES 의 지표 키(예: "general_ppl"), 값은 그 지표의 값입니다
day70_base_metrics = {"train_acc": 0.025, "domain_recall_acc": 0.036, "domain_apply_acc": 0.0, "over_refusal_rate": 0.0, "refusal_rate": 0.0, "general_acc": 0.033, "general_ppl": 36.43}

### 2-2. 어댑터 여러 개 얹기: load_adapter

베이스 모델은 한 번만 불러 두고, 교안 01의 어댑터 4개에 이름을 붙여 모두 얹은 뒤 하나씩 켜서 평가합니다. 체크포인트마다 모델 전체(약 3GB)를 불러온 지난 시간과 다른 점입니다.

- **`PeftModel.from_pretrained(베이스 모델, 폴더, adapter_name=이름)`:** 첫 어댑터를 얹어 `PeftModel`을 만듭니다. 폴더의 `adapter_config.json`대로 대상 층에 $A$, $B$를 만들고, 저장한 $A$, $B$ 값을 채웁니다.
  - 기본값 `is_trainable=False`라 어댑터 값이 학습되지 않게 고정되고, 모델은 평가 모드가 됩니다. 이어서 학습하려면 `True`로 불러옵니다.
- **`load_adapter(폴더, adapter_name=이름)`:** 같은 `PeftModel`에 어댑터를 하나 더 얹습니다. 얹기만 하고 켜지는 않으므로, 켜진 어댑터는 그대로입니다.
- **`set_adapter(이름, inference_mode=True)`:** 얹어 둔 어댑터 가운데 하나를 켭니다. 답을 만들 때는 켜진 어댑터 하나만 계산에 쓰입니다.
  - **`inference_mode`:** 켠 어댑터를 학습할 수 있게 둘지 정합니다. 기본값 `False`면 켠 어댑터를 학습할 수 있게(`requires_grad=True`) 바꾸므로, 평가만 할 때는 `True`로 고정합니다.
- **주의:** `PeftModel.from_pretrained`는 `base_model`을 복사하지 않고 그 안의 선형 층을 LoRA 층으로 바꿉니다. 그래서 이 뒤로 Base 계산은 `with lora_model.disable_adapter():` 안에서 합니다.

<img src="images/12_one_base_many_adapters.png" width="1050" alt="베이스 모델 하나에 이름표(adapter_name)를 붙인 어댑터 여러 개를 얹고 그중 하나만 켭니다. ep1은 1에폭 학습을 마치고 저장한 어댑터의 이름입니다.">

In [ ]:
from peft import PeftModel

# 교안 01이 에폭마다 저장한 어댑터 폴더 이름(checkpoint-<갱신 횟수>)에서 갱신 횟수만 꺼냅니다
# 예: checkpoint-293 -> 293. 숫자로 정렬하면 1에폭부터 차례가 됩니다
adapter_dir = output_dir / "labor_lora_lr2e-4"
steps = []
for path in adapter_dir.glob("checkpoint-*"):
    steps.append(int(path.name.split("-")[1]))
steps.sort()

# 어댑터마다 이름(ep<에폭>), 에폭, 갱신 횟수, 폴더를 한 행으로 모읍니다
# 이름은 뒤에서 set_adapter 로 어댑터를 켤 때 부르는 이름표입니다
adapter_rows = []
for epoch, step in enumerate(steps, start=1):
    adapter_rows.append({"이름": f"ep{epoch}", "에폭": epoch, "갱신 횟수": step, "폴더": adapter_dir / f"checkpoint-{step}"})

# 첫 어댑터를 얹으며 PeftModel 을 만듭니다(base_model 의 선형 층이 LoRA 층으로 바뀝니다)
first_row = adapter_rows[0]
lora_model = PeftModel.from_pretrained(base_model, first_row["폴더"], adapter_name=first_row["이름"])   # 이 어댑터의 이름표(예: "ep1")

# 나머지 어댑터를 같은 모델에 이름을 붙여 더 얹습니다. 얹기만 하고 켜지는 않습니다
for row in adapter_rows[1:]:
    lora_model.load_adapter(row["폴더"], adapter_name=row["이름"])

# peft_config 는 얹은 어댑터 이름별 설정, active_adapter 는 지금 켜진 어댑터 이름입니다(첫 어댑터 그대로)
print("갱신 횟수:", steps)
print("얹은 어댑터:", list(lora_model.peft_config.keys()), "| 켜진 어댑터:", lora_model.active_adapter)

### 2-3. 에폭별 어댑터 평가: set_adapter

어댑터를 하나씩 켜서 지표 일곱 개를 계산하고 `Base(지난 시간)` 아래에 행을 더합니다. 판사 호출은 119번 × 어댑터 4개입니다.

테이블에서는 두 신호를 봅니다.
- **외우는 신호:** 에폭이 늘수록 학습 데이터 정답률과 재현 정답률은 오르는데 적용 정답률은 떨어지면, 학습 데이터를 외우는 중입니다.
- **잊는 신호(퇴행):** 일반 질문 정답률이 떨어지거나 일반 perplexity가 오르면, 원래 능력을 잃는 중입니다.

deepeval 은 채점할 때마다 안내(WARNING, Pass Rate)를 찍습니다. 오류가 아니며, 결과는 아래 테이블에서 봅니다.

In [ ]:
# 행 이름 -> 지표 사전. Base 행을 맨 위에 두고 어댑터 행을 아래에 더합니다
model_metrics = {"Base(지난 시간)": day70_base_metrics}
epoch_tables = {}    # 어댑터 이름 -> evaluate_model 이 돌려준 채점 테이블 사전(train, domain, refusal, general, ppl)
epoch_metrics = {}   # 어댑터 이름 -> table_metrics 로 계산한 지표 일곱 개

# ep1~ep4 어댑터를 하나씩 켜서 같은 평가를 합니다
for row in adapter_rows:
    lora_model.set_adapter(row["이름"], inference_mode=True)   # 켠 어댑터를 고정한 채(학습 안 함) 씁니다
    epoch_tables[row["이름"]] = evaluate_model(lora_model)                  # 채점: 질문마다 답과 통과 여부가 든 테이블들
    epoch_metrics[row["이름"]] = table_metrics(epoch_tables[row["이름"]])   # 요약: 그 테이블들을 지표 일곱 개로(어댑터끼리 고를 때 씀)
    model_metrics[row["이름"]] = epoch_metrics[row["이름"]]                 # 같은 지표를 Base 와 한 테이블에 두려고 모든 모델 사전에도 넣음

# 행은 모델, 열은 지표가 되도록 뒤집고(.T) 열 이름을 한국어 지표 이름으로 바꿉니다
forgetting_table = pd.DataFrame(model_metrics).T.rename(columns=METRIC_NAMES)
display(forgetting_table.round(3))

**결과 읽기**
- **학습 데이터와 재현:** 학습 데이터 정답률은 0.075, 0.300, 0.375, 0.625로 에폭마다 오릅니다. 재현 정답률은 0.036~0.107(28문항 가운데 1~3개)로 에폭을 따라 오르지 않습니다.
- **적용:** 1~2에폭은 0, 3~4에폭은 0.048(21문항 가운데 1개)입니다. 더 외워도 처음 보는 사례에는 거의 답하지 못합니다.
- **일반 능력:** 일반 질문 정답률은 0.233~0.267로 Base(지난 시간)보다 높지만, 일반 perplexity는 43.3, 51.4, 56.9, 61.5로 커집니다. 정답률은 나아지고 perplexity는 나빠져 두 지표가 다른 결론을 가리키므로 함께 봅니다.

### 2-4. LoRA Tuned 고르기

지난 시간 교안 02와 같은 규칙으로 비교 기준(LoRA Tuned)을 고릅니다.
1. **학습 데이터 정답률 0.8 이상만 후보로 둡니다.** 학습이 제대로 됐다면 이미 본 답은 대부분 맞혀야 합니다.
2. **후보 가운데 노무 평가셋 통과 수(재현 + 적용)가 가장 많은 것을 고릅니다.** 같으면 앞 에폭을 고릅니다.
3. **후보가 없으면 학습 데이터 정답률이 가장 높은 것을 고릅니다.**

In [ ]:
TRAIN_ACC_MIN = 0.8   # 규칙 1: 학습 데이터 정답률이 이 값 이상인 어댑터만 후보

# 규칙 2: 후보 가운데 노무 평가셋 통과 수가 가장 많은 것(앞 에폭부터 보므로 같으면 앞 에폭)
tuned_row = None
for row in adapter_rows:
    passes = epoch_tables[row["이름"]]["domain"]["통과"].sum()   # 재현 + 적용 통과 문항 수
    if epoch_metrics[row["이름"]]["train_acc"] >= TRAIN_ACC_MIN:
        if tuned_row is None or passes > best_passes:
            tuned_row = row
            best_passes = passes

# 규칙 3: 후보가 없으면 학습 데이터 정답률이 가장 높은 것(같으면 앞 에폭)
if tuned_row is None:
    tuned_row = adapter_rows[0]
    for row in adapter_rows:
        if epoch_metrics[row["이름"]]["train_acc"] > epoch_metrics[tuned_row["이름"]]["train_acc"]:
            tuned_row = row

# 고른 어댑터의 이름, 에폭, 갱신 횟수와 테이블에 쓸 이름표(지난 시간 Tuned 이름표와 같은 모양)
# tuned_step 은 뒤의 DoRA 평가에서 같은 에폭의 체크포인트를 고를 때 씁니다
tuned_name = tuned_row["이름"]
tuned_epoch = tuned_row["에폭"]
tuned_step = tuned_row["갱신 횟수"]
tuned_label = f"LoRA Tuned {tuned_epoch}에폭(step {tuned_step})"
print("고른 어댑터:", tuned_label, "| 어댑터 이름:", tuned_name, "| 폴더:", tuned_row["폴더"])

**결과 읽기**
- **고른 어댑터:** 학습 데이터 정답률이 0.8에 이른 어댑터가 없어(가장 높은 ep4가 0.625) 규칙 3으로 ep4를 골랐습니다. 노무 평가셋 통과 수(재현 + 적용)도 ep4가 ep3과 같은 4개로 가장 많습니다(ep1 3개, ep2 1개).
- **차이 읽기:** 판사가 참일 확률을 0.5 근처로 본 문항은 답이 조금만 달라도 통과와 실패가 바뀝니다. 그래서 한두 문항 차이는 차이 없음으로 읽습니다.
- **실행마다 다른 값:** 같은 GPU 종류에서 교안 01부터 두 번 실행하면 에폭별 학습 데이터 정답률이 0.05~0.10, 도메인 통과 수가 1~3개 달라졌습니다. 학습 결과가 조금 달라지면 그리디 생성의 답 글자가 바뀌고, 판정 문장 하나라도 참일 확률이 0.5 아래로 내려가면 그 문항이 실패가 되기 때문입니다. 이 교안의 결론은 두 실행에서 같았으므로 숫자 하나보다 방향을 봅니다.

## 3. 지난 시간 SFT(풀 파인튜닝)와 비교

[Biderman et al., 2024](https://arxiv.org/abs/2405.09673)는 코드와 수학 학습 실험에서 LoRA가 풀 파인튜닝보다 도메인을 덜 배우는 대신 원래 능력도 덜 잃는 것을 보였습니다. 이 데이터에서도 그런지 같은 지표로 봅니다.

- **비교 모델:** LoRA Tuned, 지난 시간 SFT에서 가중치 전체를 학습한 풀 파인튜닝 Tuned, 같은 SFT 결과로 만든 보간 모델(Base와 마지막 체크포인트의 가중치를 α 0.75로 섞은 모델). 지난 시간 값은 아래 셀의 강사 실행 값이고, 자기 값이 있으면 바꿔 넣습니다.
- **주의:** 지난 시간 SFT(풀 파인튜닝)는 퇴행을 줄이려고 낮은 학습률(lr 5e-6)로 학습했고, LoRA Tuned는 lr 2e-4입니다. 그래서 두 모델의 차이에는 학습 방법(LoRA, 풀 파인튜닝)뿐 아니라 학습률 차이도 섞여 있습니다.
- **볼 것:** 도메인 지표로 얼마나 배웠는지, 일반 질문 정답률과 일반 perplexity 변화율(Base보다 일반 perplexity가 몇 % 커졌는지)로 얼마나 잃었는지.

In [ ]:
# 지난 시간 교안 02 지표 테이블(summary_table)의 Tuned 행과 보간 행입니다(강사 실행 값, 자기 결과가 있으면 바꿔 넣습니다)
# 키는 day70_base_metrics 와 같은 METRIC_NAMES 의 지표 키입니다
day70_full_ft_metrics = {"train_acc": 0.85, "domain_recall_acc": 0.286, "domain_apply_acc": 0.048, "over_refusal_rate": 0.0, "refusal_rate": 0.333, "general_acc": 0.167, "general_ppl": 59.71}
day70_interp_metrics = {"train_acc": 0.725, "domain_recall_acc": 0.214, "domain_apply_acc": 0.0, "over_refusal_rate": 0.0, "refusal_rate": 0.25, "general_acc": 0.233, "general_ppl": 48.41}

full_compare_table = pd.DataFrame({
    "Base(지난 시간)": day70_base_metrics,
    tuned_label: epoch_metrics[tuned_name],
    "풀 파인튜닝 Tuned(지난 시간)": day70_full_ft_metrics,
    "보간 α=0.75(지난 시간)": day70_interp_metrics,
}).T.rename(columns=METRIC_NAMES)

# Base 대비 일반 perplexity 변화율(%) = (모델 perplexity / Base perplexity - 1) * 100. 양수면 일반 문단을 Base보다 못 예측합니다
# 예: Base 40, 모델 60 이면 (60 / 40 - 1) * 100 = +50%
base_general_ppl = full_compare_table.loc["Base(지난 시간)", "일반 perplexity"]
full_compare_table["일반 perplexity 변화율(%)"] = (full_compare_table["일반 perplexity"] / base_general_ppl - 1) * 100
display(full_compare_table.round(3))

**결과 읽기**
- **도메인:** LoRA Tuned는 학습 데이터 정답률(0.625)과 재현 정답률(0.107)이 풀 파인튜닝 Tuned(지난 시간)보다 낮고, 적용 정답률은 같습니다(21문항 가운데 1개). 거절률 0.250(범위 밖 12개 가운데 3개)은 한 문항 적습니다. LoRA가 덜 배웠습니다.
- **일반 능력:** 일반 perplexity 변화율은 +68.8%로 풀 파인튜닝 Tuned와 5%p 안쪽이고, 일반 질문 정답률은 30문항 가운데 2개 많습니다.
- **보간 행:** 일반 perplexity 변화율이 LoRA Tuned의 절반쯤이고 학습 데이터와 재현 정답률도 높으며, 일반 질문 정답률은 같습니다. LoRA Tuned가 나은 것은 적용 한 문항뿐입니다.
- **논문 결과와 같은지:** '덜 배운다'는 맞고 '덜 잊는다'는 맞지 않았습니다. 비교한 풀 파인튜닝이 낮은 학습률로 퇴행을 줄인 모델이라는 조건이 붙습니다. 퇴행이 걱정이면 LoRA로 바꾸는 것만으로는 부족하고, 학습한 모델마다 일반 질문과 일반 문단으로 일반 능력을 직접 평가해야 합니다.

## 4. DoRA: use_dora

### 4-1. 가중치를 크기와 방향으로 나누기

**DoRA(Weight-Decomposed Low-Rank Adaptation, 가중치 분해 저랭크 적응)**: 가중치를 크기(magnitude)와 방향(direction)으로 나눠, 크기 $m$은 직접 학습하고 방향은 LoRA($BA$)로 조정하는 PEFT 방법입니다([Liu et al., 2024](https://arxiv.org/abs/2402.09353)).
- **크기 $m$:** 가중치 행렬의 행(출력 하나)마다 그 행의 길이(노름)를 하나씩 담은 벡터입니다. 출력이 1,024개인 층이면 $m$도 값 1,024개입니다.
- **방향:** 각 행을 자기 길이로 나눠 길이를 1로 맞춘 행렬입니다. 크기 정보가 빠지고 방향만 남습니다.
- **목적:** 크기와 방향을 따로 바꿀 수 있게 해 풀 파인튜닝이 가중치를 바꾸는 방식에 더 가깝게 학습하고, 같은 rank에서 LoRA보다 나아지려는 방법입니다.
- **쓰는 법:** peft에서는 `LoraConfig`에 `use_dora=True` 한 줄을 더합니다.

노름(길이)은 출력 행마다 계산합니다.

$\displaystyle W' = m \cdot \frac{W + \frac{\alpha}{r} B A}{\lVert W + \frac{\alpha}{r} B A \rVert}$

- **$\frac{\alpha}{r}BA$:** LoRA와 같은 어댑터 변화량입니다.
- **$m$:** $W$의 행별 길이로 시작합니다. LoRA처럼 $B$가 0으로 시작해 $BA = 0$이므로, DoRA 어댑터를 붙인 직후의 $W'$는 $W$와 같고 출력도 베이스 모델과 같습니다.
- **빼는 층:** 0번 레이어의 `layers.0.linear_attn.in_proj_qkv` 층 가중치에는 값이 모두 0인 행이 있습니다. 그 행은 길이가 0이라 $m$이 0으로 시작하고, 나누기가 0/0이 되어 첫 단계부터 손실이 NaN이 됩니다.
- **`exclude_modules`:** `target_modules`로 고른 층 가운데 어댑터를 붙이지 않을 층 이름을 주는 `LoraConfig` 인자입니다. 이 층 하나만 빼고 나머지 185개 층에 DoRA 어댑터를 붙입니다.

**LoRA와 비교**

| 비교 | LoRA | DoRA |
|---|---|---|
| 학습하는 것 | 가중치에 더할 양 $BA$ | 크기 $m$(출력 행마다 하나) + 방향을 조정하는 $BA$ |
| 학습 파라미터 수 | $A$, $B$ | $A$, $B$ + $m$(이 모델에서 LoRA보다 336,448개 많음) |
| 학습 속도 | 기준 | 매 단계 행 길이(노름) 계산이 더해져 LoRA보다 느림 |
| 추론 | 병합하면 어댑터 계산 없음 | 병합하면 LoRA와 같이 어댑터 계산 없음 |

- **논문 결과:** 논문 실험에서 DoRA는 같은 rank의 LoRA보다 LLaMA 상식 추론 등 여러 과제에서 꾸준히 높은 점수를 냈습니다. PEFT 문서 [LoRA](https://huggingface.co/docs/peft/developer_guides/lora)는 DoRA가 특히 `r`이 작을 때 LoRA보다 나을 수 있다고 하고, LoRA보다 계산 부담이 커서 추론할 때는 병합하기를 권합니다.
- **실무에서 언제:** LoRA 결과가 풀 파인튜닝보다 부족하고 학습 시간을 더 쓸 수 있을 때, 또는 `r`을 작게 유지하고 싶을 때 시도합니다.
- **이 데이터에서:** 결과는 4-4 비교 테이블에서 LoRA Tuned와 직접 비교합니다.

<img src="images/14_dora.png" width="1050" alt="DoRA는 가중치를 크기 m(학습)과 방향(길이 1)으로 나누고, 방향을 LoRA의 B·A로 조정합니다: W' = m · (W + ΔW) / ‖W + ΔW‖.">

### 4-2. DoRA 학습

LoRA Tuned의 `LoraConfig`에 `use_dora=True`와, 앞에서 본 층 하나를 빼는 `exclude_modules`만 더해 학습합니다.

- **같게 두는 것:** 학습률(lr 2e-4), 4에폭 일정, 데이터, 시드. 평가는 LoRA Tuned와 같은 에폭(`tuned_step`)의 체크포인트로 합니다.
- **새 베이스 모델에 붙이기:** `lora_model`의 베이스에는 이미 어댑터 4개가 얹혀 있으므로, 학습 전 모델을 새로 불러와 DoRA 어댑터를 붙입니다.
- **학습 시간:** 학습기의 `train_runtime`(초)을 교안 01 학습(MLflow 기록)과 비교합니다. 같은 GPU 종류끼리만 비교합니다.

In [ ]:
import gc

from datasets import Dataset
from peft import LoraConfig, get_peft_model
from trl import SFTConfig, SFTTrainer

# 교안 01과 같은 학습, 검증 Dataset 입니다(train_rows 는 앞에서 읽었습니다)
# to_prompt_completion 은 한 행을 SFTTrainer 가 읽는 prompt, completion 메시지 모양으로 바꿉니다
val_rows = load_jsonl(data_dir / "본실습" / "labor_val.jsonl")
train_dataset = Dataset.from_list([to_prompt_completion(row) for row in train_rows])
val_dataset = Dataset.from_list([to_prompt_completion(row) for row in val_rows])

# 바꾸는 것: use_dora=True. 나머지 값은 LoRA Tuned 와 같습니다
dora_config = LoraConfig(r=8, lora_alpha=16, target_modules="all-linear", lora_dropout=0.05, task_type="CAUSAL_LM",
                         use_dora=True,                           # 층마다 크기 m 을 더 두고, BA 로는 방향을 조정합니다
                         exclude_modules=["layers.0.linear_attn.in_proj_qkv"])   # 값이 모두 0인 행이 있는 층 하나에는 어댑터를 붙이지 않습니다

# 학습 전 모델을 새로 불러와 DoRA 어댑터를 붙입니다(B 가 0이라 붙인 직후 출력은 학습 전 모델과 같습니다)
dora_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32).to(device)
dora_model = get_peft_model(dora_base, dora_config)

# 교안 01 의 설정과 같은 값입니다. save_strategy="epoch" 라 에폭마다 어댑터 폴더가 하나씩 생깁니다
dora_args = SFTConfig(
    output_dir=str(output_dir / "labor_dora"),
    learning_rate=2e-4,
    num_train_epochs=4,
    per_device_train_batch_size=4,
    seed=0,
    gradient_checkpointing=False,
    max_length=1024,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_only_model=True,
    logging_steps=50,
    eval_on_start=True,
    report_to=["mlflow"],
    run_name="dora",
)
dora_trainer = SFTTrainer(model=dora_model, args=dora_args, train_dataset=train_dataset,
                          eval_dataset=val_dataset, processing_class=tokenizer)
# run 하나에 설정(r, lora_alpha, 대상 층, DoRA 여부)을 기록하고 학습합니다. 손실은 학습기가 같은 run 에 남깁니다
# get_peft_model 이 target_modules 를 실제 층 이름 목록으로 바꿔 두므로 대상 층은 지정한 값을 적습니다
with mlflow.start_run(run_name="dora"):
    mlflow.log_params({"lora_r": dora_config.r, "lora_alpha": dora_config.lora_alpha, "target_modules": "all-linear",
                       "use_dora": dora_config.use_dora})
    dora_output = dora_trainer.train()

# 학습기와 모델을 지워 GPU 메모리를 비웁니다(어댑터는 에폭마다 저장했습니다)
del dora_trainer, dora_model, dora_base
gc.collect()
torch.cuda.empty_cache()

### 4-3. DoRA 평가

LoRA Tuned와 같은 에폭의 DoRA 어댑터를 이름 `dora`로 `lora_model`에 더 얹어 같은 지표로 평가합니다. DoRA 어댑터도 같은 베이스 모델로 학습했으므로 그대로 얹을 수 있습니다.

In [ ]:
# LoRA Tuned 와 같은 갱신 횟수(tuned_step)의 DoRA 체크포인트를 골라 같은 학습량끼리 비교합니다
dora_dir = output_dir / "labor_dora" / f"checkpoint-{tuned_step}"

# DoRA 어댑터를 같은 lora_model 에 이름 "dora"로 더 얹고, 켜서 같은 평가를 합니다
lora_model.load_adapter(dora_dir, adapter_name="dora")
lora_model.set_adapter("dora", inference_mode=True)
dora_tables = evaluate_model(lora_model)
dora_metrics = table_metrics(dora_tables)

### 4-4. LoRA Tuned와 DoRA 비교 테이블

LoRA Tuned와 DoRA의 학습 파라미터 수, 학습 시간, 지표 일곱 개를 한 테이블로 봅니다. 학습 파라미터 수는 어댑터 파일(`adapter_model.safetensors`)에 저장된 값의 개수입니다. 이 파일에는 학습한 값만 있습니다.

In [ ]:
from safetensors.torch import load_file

# LoRA Tuned 의 학습 시간: 교안 01 run 에 학습기가 남긴 train_runtime(초)
# search_runs 는 run 하나를 한 행으로 한 DataFrame 을 돌려줍니다. 같은 이름의 run 이 여러 개면 가장 최근 것(맨 위)을 씁니다
runs_table = mlflow.search_runs(experiment_names=["labor-lora"], order_by=["start_time DESC"])
tuned_run = runs_table[runs_table["tags.mlflow.runName"] == "lora_lr_2e-4"].iloc[0]
tuned_seconds = round(tuned_run["metrics.train_runtime"])

# 설정마다 행 이름, 어댑터 폴더, 학습 시간(DoRA 는 학습기가 잰 전체 학습 시간)
setting_sources = [(tuned_label, tuned_row["폴더"], tuned_seconds), ("DoRA", dora_dir, round(dora_output.metrics["train_runtime"]))]

# 학습 파라미터 수: 어댑터 파일에 든 텐서마다 값의 개수(numel)를 더합니다
# 파일에는 학습한 A, B 와 DoRA 의 크기 m 만 있습니다. 예: 모양 (8, 1024) 인 A 하나는 8 * 1024 = 8,192개
setting_info = []
for label, folder, seconds in setting_sources:
    param_count = 0
    for tensor in load_file(folder / "adapter_model.safetensors").values():
        param_count += tensor.numel()
    setting_info.append({"설정": label, "학습 파라미터 수": param_count, "학습 시간(초)": seconds})

# 파라미터 수, 학습 시간 테이블 옆에 지표 일곱 개를 설정 이름(행)으로 맞춰 붙입니다
setting_metrics = {tuned_label: epoch_metrics[tuned_name], "DoRA": dora_metrics}
setting_table = pd.DataFrame(setting_info).set_index("설정").join(pd.DataFrame(setting_metrics).T.rename(columns=METRIC_NAMES))
display(setting_table.round(3))

**결과 읽기**
- **학습 파라미터 수:** DoRA는 5,747,776개로 LoRA Tuned(5,411,328개)보다 336,448개 많습니다. 크기 $m$은 DoRA 어댑터를 붙인 185개 층의 출력 크기(`out_features`) 합인 393,792개이고, 여기서 뺀 층의 $A$, $B$ 57,344개를 빼면 두 값의 차이와 같습니다.
- **학습 시간:** LoRA Tuned 422초, DoRA 596초(1.4배)입니다. 매 단계 행 길이(노름)를 계산하는 만큼 느립니다. 같은 GPU 종류라도 파드마다 속도가 달라(다른 파드에서는 356초, 505초) 같은 실행 안의 배율로 비교합니다.
- **DoRA 지표:** 학습 데이터 정답률 0.625와 도메인 통과 수 4개(재현 4, 적용 0)가 LoRA Tuned와 같고, 일반 질문 정답률은 한 문항 적으며 과잉 거절이 3문항 생겼습니다. 일반 perplexity는 59.6으로 비슷합니다. 지표는 나아지지 않고 학습 시간만 1.4배라 DoRA로 바꿀 근거가 없습니다.
- **시드:** 두 어댑터 모두 한 시드로 한 번씩 학습했습니다. 차이가 작으면 시드를 바꿔 같은 방향인지 확인한 뒤 결론을 씁니다.

## 5. 쓸 어댑터 정하기

풀 파인튜닝 비교 테이블(`full_compare_table`)과 DoRA 비교 테이블(`setting_table`)로 노무 상담에 쓸 모델을 정합니다. 결정 테이블에는 결정, 근거, 다시 검토할 조건을 적습니다.

**결정 테이블**

| 결정 | 근거 | 다시 검토할 조건 |
|---|---|---|
| LoRA Tuned(4에폭, r 8, all-linear)를 씁니다. | DoRA: 지표는 같고 학습이 1.4배 느립니다.<br>보간 α=0.75(지난 시간): 지표는 비슷하거나 낫지만 분야마다 모델 전체(약 3GB)가 필요해 어댑터를 바꿔 끼울 수 없습니다. | 적용 정답률을 올려야 하면 -> 조문을 찾아 붙이는 RAG를 더해 다시 평가합니다.<br>도메인 정답률을 더 올려야 하면 -> r, 학습률, 에폭을 한 번에 하나씩 바꿔 다시 학습하고 같은 평가셋으로 비교합니다. |

## 6. 정한 어댑터 병합과 저장

### 6-1. 병합: merge_and_unload

어댑터를 따로 두면 답을 만들 때마다 어댑터 계산이 더해집니다. 바꿔 끼울 일이 없다면 어댑터를 베이스 가중치에 미리 더해, 어댑터가 없는 보통 Transformers 모델(`PeftModel`이 아님) 하나로 만듭니다. 이렇게 만든 모델을 병합 모델이라고 부릅니다.

**병합(merge)**
- LoRA 층의 출력은 $Wx + \frac{\alpha}{r}BAx = (W + \frac{\alpha}{r}BA)x$이므로, 아래 $W'$를 $W$ 자리에 넣으면 어댑터 없이 같은 계산이 됩니다.

$\displaystyle W' = W + \frac{\alpha}{r}\, B A$

- **`merge_and_unload()`:** 대상 층마다 $W'$를 넣고 LoRA 층을 원래 선형 층으로 되돌린 모델을 돌려줍니다.
- **새 베이스 모델에 병합하는 이유:** `lora_model`에서 병합하면 그 안의 베이스 가중치가 바뀌고 LoRA 층이 사라져, 얹어 둔 다른 어댑터와 어댑터를 끈 Base 계산(함께 따라하기에서 씀)을 쓸 수 없습니다. 그래서 새로 불러온 베이스 모델에 LoRA Tuned 어댑터(`tuned_row["폴더"]`) 하나만 얹어 병합합니다.

<img src="images/13_merge_tradeoff.png" width="1050" alt="merge_and_unload는 어댑터를 W에 더해 W' = W + (α / r) B A 하나로 만듭니다. 어댑터로 두면 바꿔 끼우기와 끄기가 되고, 병합하면 추론에 어댑터 계산이 없습니다.">

In [ ]:
# 새로 불러온 베이스 모델에 정한 어댑터 하나만 얹고 병합합니다(lora_model 의 베이스와 어댑터는 그대로 둡니다)
# merge_and_unload 는 대상 층마다 W 자리에 W + (lora_alpha / r) * B A 를 넣고 LoRA 층을 걷어 낸 모델을 돌려줍니다
merge_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32).to(device)
merged_model = PeftModel.from_pretrained(merge_base, tuned_row["폴더"]).merge_and_unload()

# 병합 모델은 PeftModel 이 아닌 보통 모델이고, gate_proj 도 어댑터가 없는 원래 선형 층입니다
print("병합 모델:", type(merged_model).__name__, "| 첫 층 gate_proj:", merged_model.model.layers[0].mlp.gate_proj)

### 6-2. 병합 전과 후의 답 비교

노무 평가셋 전체를 병합 모델로 다시 생성해, LoRA Tuned 어댑터가 낸 답(`epoch_tables[tuned_name]["domain"]["답변"]`)과 글자까지 같은지 셉니다.

In [ ]:
# 노무 평가셋의 질문만 모아, 병합 모델로 앞과 같은 그리디 생성을 합니다
eval_questions = []
for row in eval_rows:
    eval_questions.append(row["question"])
merged_answers = generate_replies(merged_model, eval_questions, SYSTEM_PROMPT)

# 같은 질문 순서로 짝지어, 정한 어댑터의 답(앞에서 채점한 테이블)과 글자까지 같은 답의 수를 셉니다
same_answer_count = 0
for adapter_answer, merged_answer in zip(epoch_tables[tuned_name]["domain"]["답변"], merged_answers):
    if adapter_answer == merged_answer:
        same_answer_count += 1
print("같은 답:", same_answer_count, "/", len(merged_answers))

**결과 읽기**
- **답:** 49문항 모두 글자까지 같습니다.
- **그래도 비교하는 이유:** 병합 전($Wx$와 $\frac{\alpha}{r}BAx$를 따로 계산해 더함)과 병합 후($W'x$)는 계산 순서가 달라 부동소수점 반올림 오차가 조금 다릅니다. 그래서 다음 토큰 후보 가운데 1등과 2등 점수가 거의 같은 자리에서는 다른 토큰이 골라질 수 있습니다.

### 6-3. 병합 모델 저장: save_pretrained

병합 모델은 어댑터가 없는 보통 모델이라, 지난 시간 풀 파인튜닝 모델처럼 모델 전체를 토크나이저와 함께 저장합니다.

In [ ]:
# 병합 모델과 토크나이저를 한 폴더에 저장합니다(float32 모델 전체)
merged_dir = output_dir / "labor_lora_merged"
merged_model.save_pretrained(merged_dir)
tokenizer.save_pretrained(merged_dir)

# 가중치 파일끼리 크기(MB)를 비교합니다(폴더에는 토크나이저 파일 등도 있습니다)
# st_size 는 바이트 수라 1024 ** 2 로 나눠 MB 로 바꿉니다
size_table = pd.DataFrame([
    {"저장물": "고른 어댑터", "크기(MB)": round((tuned_row["폴더"] / "adapter_model.safetensors").stat().st_size / 1024 ** 2, 1)},
    {"저장물": "병합 모델", "크기(MB)": round((merged_dir / "model.safetensors").stat().st_size / 1024 ** 2, 1)},
])
display(size_table)

# 다 쓴 병합 모델을 지워 GPU 메모리를 비웁니다
del merged_model, merge_base
gc.collect()
torch.cuda.empty_cache()

**결과 읽기**
- 병합 모델 가중치 파일은 2,870.2MB(float32 모델 전체, 약 3GB)이고, 어댑터 파일은 20.7MB로 그 약 0.7%입니다.

### 6-4. 어댑터 유지와 병합 비교

| 비교 | 어댑터로 둘 때 | 병합할 때 |
|---|---|---|
| 저장 크기 | 어댑터만(베이스 모델은 따로 하나) | 분야마다 모델 전체 |
| 바꿔 끼우기 | 베이스 모델 하나에 여러 어댑터를 얹고 골라 켬 | 할 수 없음. 분야마다 모델이 따로 필요 |
| 어댑터 끄기 | `disable_adapter()`로 학습 전 모델과 같은 계산 | 할 수 없음. 학습 전 모델을 따로 둠 |
| 추론 계산 | 층마다 어댑터 계산이 더해짐 | 어댑터 계산이 없음 |
| 이어서 학습 | `is_trainable=True`로 불러와 이어서 학습 | 병합 모델에서는 어댑터를 떼어 낼 수 없음. 보관한 어댑터 폴더를 베이스 모델에 다시 얹어 학습 |

- 분야가 여럿이거나 어댑터를 자주 바꾸면 어댑터로 두고, 한 분야만 운영하며 어댑터 계산을 없애려면 병합합니다. 병합해도 어댑터 폴더는 보관합니다.

## 7. 🖐️ 함께 따라하기: 전월세 어댑터 평가와 어댑터 바꿔 끼우기

교안 01 따라하기에서 저장한 전월세 어댑터를 `lora_model`에 더 얹어 평가하고, 노무 어댑터와 바꿔 가며 답하게 합니다.

**본실습과 다른 점**
- **어댑터:** `output/lease_lora_lr2e-4`를 이름 `lease`로 얹습니다. 이 폴더가 없으면(교안 01 함께 따라하기를 끝내지 못했으면) 교안 01을 새 커널에서 처음부터 함께 따라하기 끝까지 실행해 만듭니다.
- **평가셋과 시스템 프롬프트:** `data/따라하기/lease_eval.jsonl`(정답 답변과 근거 조문), `SYSTEM_PROMPT_LEASE`. 재현과 적용을 나누지 않고 도메인 정답률 하나로 봅니다.
- **비교 행:** 지난 시간 따라하기의 Base와 전월세 풀 파인튜닝(lr 5e-6, 4에폭) 도메인 정답률입니다. Base는 다시 채점하지 않습니다. 코드의 강사 실행 값은 자기 값이 있으면 바꿔 넣습니다.

**실습 진행 방법:** 작성 셀의 `...`만 주석에 맞춰 채웁니다(어댑터 얹기, 켜기, 결과 해석의 내 판단 두 줄). 먼저 직접 작성해 봅니다. 완성 코드는 수업이 끝난 뒤 강사가 공유합니다.

### 7-1. 전월세 어댑터 얹기: load_adapter

In [ ]:
SYSTEM_PROMPT_LEASE = "너는 전월세 상담 챗봇이다. 주택임대차보호법 조문에 근거해 존댓말로 간결히 답한다."

# 전월세 평가셋: 질문, 정답 답변, 근거 조문(law_text)
lease_eval_rows = load_jsonl(data_dir / "따라하기" / "lease_eval.jsonl")

# 🖐️ 함께 따라하기: 교안 01 따라하기에서 저장한 전월세 어댑터를 lora_model 에 이름 "lease"로 더 얹으세요
# 폴더는 output_dir / "lease_lora_lr2e-4" 입니다. 얹기만 하고 켜지는 않습니다
...

print("평가 문항 수:", len(lease_eval_rows), "| 얹은 어댑터:", list(lora_model.peft_config.keys()))

### 7-2. 전월세 평가셋 채점

In [ ]:
# 🖐️ 함께 따라하기: 전월세 어댑터를 켜고 같은 평가셋을 채점하세요
lora_model.set_adapter(..., inference_mode=True)
lease_table = answer_and_judge(lora_model, lease_eval_rows, SYSTEM_PROMPT_LEASE)

# 도메인 정답률은 통과 열(True/False)의 평균입니다. 지난 시간 행은 강사 실행 값입니다(자기 값이 있으면 바꿔 넣습니다)
lease_compare_table = pd.DataFrame([
    {"모델": "Base(지난 시간)", "도메인 정답률": 0.0},
    {"모델": "전월세 어댑터", "도메인 정답률": lease_table["통과"].mean()},
    {"모델": "전월세 풀 파인튜닝(지난 시간)", "도메인 정답률": 0.25},
]).set_index("모델")
display(lease_compare_table.round(3))

### 7-3. 어댑터 바꿔 끼우기: set_adapter

노무 질문과 전월세 질문 하나씩을 노무 어댑터(LoRA Tuned), 전월세 어댑터, 어댑터를 끈 Base로 답하게 합니다. 시스템 프롬프트는 질문 분야에 맞춥니다.

In [ ]:
# (분야, 질문, 그 분야의 시스템 프롬프트) 묶음입니다. 분야마다 평가셋 첫 질문 하나를 씁니다
switch_questions = [("노무", eval_rows[0]["question"], SYSTEM_PROMPT), ("전월세", lease_eval_rows[0]["question"], SYSTEM_PROMPT_LEASE)]

switch_rows = []
for domain, question, system_prompt in switch_questions:

    # 🖐️ 함께 따라하기: 노무 어댑터(tuned_name)를 켜고 답하게 하세요
    lora_model.set_adapter(..., inference_mode=True)
    labor_answer = generate_replies(lora_model, [question], system_prompt)[0]

    # 🖐️ 함께 따라하기: 전월세 어댑터를 켜고 답하게 하세요
    lora_model.set_adapter(..., inference_mode=True)
    lease_answer = generate_replies(lora_model, [question], system_prompt)[0]

    # with 블록 안에서만 어댑터를 모두 끄고 Base 로 답합니다. 블록을 나가면 켜져 있던 어댑터가 다시 켜집니다
    with lora_model.disable_adapter():
        off_answer = generate_replies(lora_model, [question], system_prompt)[0]

    switch_rows.append({"분야": domain, "질문": question, "노무 어댑터(LoRA Tuned)": labor_answer,
                        "전월세 어댑터": lease_answer, "어댑터 끔(Base)": off_answer})
display(pd.DataFrame(switch_rows))

### 7-4. 결과 해석

앞의 두 테이블로 두 가지를 판단합니다.
- 전월세 어댑터의 도메인 정답률이 Base(지난 시간), 지난 시간 전월세 풀 파인튜닝과 비교해 어떤지
- 베이스 모델 하나에서 어댑터만 바꿔 각 분야의 말투로 답하는지

**내 판단**
- **도메인 정답률:**
- **바꿔 끼우기:**

## 8. 핵심 정리

- **여러 어댑터 얹기:** `PeftModel.from_pretrained`로 얹고, `load_adapter`로 더 얹고, `set_adapter`로 하나를 켭니다. 베이스 모델은 한 번만 불러옵니다. Base는 답이 같아 지난 시간 값을 썼습니다.
- **LoRA Tuned:** 학습 데이터 정답률이 가장 높은 4에폭(step 1172, 0.625)을 골랐습니다. 도메인 통과 수도 4개로 가장 많았습니다(3에폭과 같음).
- **풀 파인튜닝과 비교:** 학습 데이터와 재현 정답률이 낮아 덜 배웠지만, 일반 perplexity 변화율(+68.8%)은 비슷해 덜 잊지는 않았습니다. 지난 시간 보간 모델이 적용 한 문항을 빼고 같거나 나았습니다. 비교한 풀 파인튜닝은 낮은 학습률(lr 5e-6)로 퇴행을 줄인 모델입니다.
- **DoRA:** 가중치를 크기 $m$과 방향으로 나눠 학습합니다. 파라미터 5,747,776개(LoRA Tuned보다 336,448개 많음), 학습 시간 1.4배로 학습 데이터 정답률과 도메인 통과 수가 LoRA Tuned와 같았습니다.
- **결정과 병합:** 정한 어댑터를 `merge_and_unload`로 $W' = W + \frac{\alpha}{r}BA$처럼 미리 더하면 어댑터 계산이 없는 보통 모델이 됩니다. 답은 49문항 모두 같았고, 가중치 파일은 병합 모델 2,870.2MB, 어댑터 20.7MB입니다.

다음 시간에는 베이스 모델을 4비트로 불러와 메모리를 줄인 채 어댑터를 학습하는 QLoRA를 배웁니다.